# Behind the scenes: Day Two
### Data download and preparation

1. Beck et al. ALREADY DONE IN DAY1_BTS!
2. GBIF species occurrence records (Philippine Eagle, Flying Fox) + a broader mammal community dataset
3. Sentinel-2 satellite imagery for Negros island
4. Hansen Global Forest Change data for Negros

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray

from workshop_utils import RAW_DIR, PROCESSED_DIR
from workshop_utils.download import (
    download_gbif, download_gbif_compilation, download_hansen_tiles,
    search_seasonal_scenes, build_landsat_sentinel_composite,
)
print(f'RAW_DIR: {RAW_DIR}')

---
## 1. Beck et al. (2023) — Köppen-Geiger maps and 1 km climate data

> Beck, H. E., et al. (2023). High-resolution (1 km) Köppen-Geiger maps

Already done!

---
## 2. GBIF species occurrence records

Two species, two purposes:
- **Philippine Eagle** (*Pithecophaga jefferyi*, GBIF taxonKey 2480381) — Day 2 biodiversity showcase.
  Even with few records, the map tells a powerful conservation story.
- **Philippine Flying Fox** (*Pteropus vampyrus*, GBIF taxonKey 5218644) — Day 3 SDM target.
  Widespread across the archipelago, with enough records for a meaningful model.

⚠️ Verify taxonKeys via `GET /v1/species/match?name=...` before use — GBIF backbone taxonomy
keys can go stale (the previous keys for both species silently returned 0 PH records).

No API key required. Pagination via `offset`.

In [ ]:
eagle = download_gbif(
    taxon_key=2480381,  # Pithecophaga jefferyi (verified via /v1/species/match; old key 2480965 was stale, returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'philippine_eagle.csv',
)
eagle.head()

In [ ]:
fox = download_gbif(
    taxon_key=5218644,  # Pteropus vampyrus (old key 2432859 was stale and returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'flying_fox.csv',
)
print(fox['year'].value_counts().sort_index().tail(15))
fox.head()

In [ ]:
# --- Clean + copy to PROCESSED_DIR/day2 for student use ---
day2_dir = PROCESSED_DIR / "day2"
day2_dir.mkdir(parents=True, exist_ok=True)

for label, fname in [("philippine_eagle", "philippine_eagle.csv"), ("flying_fox", "flying_fox.csv")]:
    df = pd.read_csv(RAW_DIR / "GBIF" / fname)
    df = (
        df.dropna(subset=["lon", "lat"])
          .drop_duplicates(subset=["gbifID"])
          .query("114 <= lon <= 127 and 4 <= lat <= 22")  # sanity-check: inside PH bounds
          .sort_values("year")
    )
    df.to_csv(day2_dir / f"gbif_{label}.csv", index=False)
    print(f"saved gbif_{label}.csv ({len(df)} rows)")

### 2b. GBIF mammal community dataset — urbanisation & terrestrial mammals in the Philippines

A much richer complement to the two single-species pulls above: a compiled GBIF occurrence
download of **all present-day terrestrial mammal records for the Philippines**
(class Mammalia, `COUNTRY=PH`, `OCCURRENCE_STATUS=present`), assembled from iNaturalist,
museum specimens, barcoding projects, etc. Backs the paper *"Making it in the city: Public
data reveal patterns and drivers of terrestrial mammal occurrence across urban Philippines."*

- 67,250 raw records → 289 species after cleaning
- Includes IUCN Red List category, island/province/municipality, and ~5,500 Negros-tagged
  records — good direct overlap with our Sentinel-2 / Hansen Negros data for Part IV
  (urban expansion vs. mammal occurrence)
- GBIF download DOI: [10.15468/dl.6ju8sc](https://doi.org/10.15468/dl.6ju8sc) — **cite this
  DOI (see `citations.txt` in the download) whenever the dataset is used**, since it is a
  compilation of many constituent datasets.

In [ ]:
GBIF_DOWNLOAD_KEY = "0003657-260226173443078"
KEEP_COLS = [
    "gbifID", "species", "genus", "family", "order", "class",
    "decimalLongitude", "decimalLatitude", "coordinateUncertaintyInMeters",
    "year", "month", "day", "basisOfRecord", "individualCount",
    "island", "islandGroup", "stateProvince", "county", "municipality", "locality",
    "iucnRedListCategory", "establishmentMeans", "institutionCode", "datasetKey",
]

mammals = download_gbif_compilation(GBIF_DOWNLOAD_KEY, KEEP_COLS)
before = len(mammals)
mammals = mammals.dropna(subset=["decimalLongitude", "decimalLatitude", "species"])
mammals = mammals[mammals["class"] == "Mammalia"]
print(f"{before} -> {len(mammals)} rows after cleaning  ({mammals['species'].nunique()} species)")
print(mammals["iucnRedListCategory"].value_counts(dropna=False))

negros_mask = (mammals["county"].astype(str).str.contains("NEGROS", case=False, na=False) |
               mammals["stateProvince"].astype(str).str.contains("Negros", case=False, na=False))
print(f"Negros-tagged records: {negros_mask.sum()}")

In [ ]:
# --- Save to PROCESSED_DIR/day2 for student use ---
out_csv = day2_dir / "gbif_mammals_philippines.csv"
mammals.to_csv(out_csv, index=False)
print(f"saved {out_csv}  ({out_csv.stat().st_size / 1e6:.1f} MB)")

---
## 3. Multispectral imagery for land-cover comparison — Negros

Two composites, built to actually be comparable:
- **Historical**: **Landsat 5 TM**, ~1990-1996
- **Recent**: **Sentinel-2 L2A**, ~2022-2024

That ~32-year gap (vs. a single 2018-2024 pull) gives students a genuinely dramatic, teachable
land-use-change signal instead of noise from a 6-year window.

⚠️ Two problems with grabbing "the lowest-cloud scene per tile" (the original approach):
1. **Clouds/shadows leave real gaps** even in the least-cloudy single scene available — there's no
   fallback data to fill them.
2. **Season contaminates the comparison.** Comparing scenes from different months of the year (e.g.
   Feb vs. April) confounds seasonal NDVI/greenness shifts with real land-cover change.

Fix for both: instead of one scene, pool **many** scenes over a **fixed season** across **several
years**, mask clouds/shadows per-pixel using each sensor's own QA band, and take the **per-pixel
temporal median**. Gaps in one scene get filled by another date; season is held constant; outlier
cloud/shadow pixels get median'd away.

We use the **Philippine dry season (Feb-Apr)** — confirmed empirically below to be far less cloudy
over Negros than the wet-season months (median cloud cover ~18% vs. ~53% for Jun-Aug).

Sources — both public STAC catalogs, no credentials needed:
- Sentinel-2 L2A: Element84 Earth Search (`sentinel-2-l2a`), using its `scl` (Scene Classification
  Layer) band for cloud/shadow masking
- Landsat 5 TM: Microsoft Planetary Computer (`landsat-c2-l2`), using its `qa_pixel` band (bit 6 =
  "clear") for masking; DN converted to physical surface reflectance via the official USGS scale/offset
  (×0.0000275, −0.2) — unlike Sentinel-2's pure multiplicative reflectance scale, Landsat Collection 2's
  additive offset means NDVI computed on raw DN would be wrong

Both are read via a **fast windowed + decimated native-resolution read** first (exploiting each COG's
built-in overviews), *then* reprojected onto one common ~100 m lat/lon grid so every scene — any tile,
any date, either sensor — stacks pixel-for-pixel before compositing. (Reprojecting directly against the
remote COG instead, e.g. via a naive `WarpedVRT`, turned out ~4x slower in testing — it bypasses the
overviews GDAL would otherwise use.)

Output:
- `data/raw/Sentinel2/negros_imagery.nc` — full composite, ~100 m
- `data/processed/day2/negros_imagery.nc` — coarsened ~200 m, exercise-ready

Note: this is a **separate file** from `negros_sentinel2.nc` (the original single-scene 2018/2024
pull), which is left untouched — Part IV of `day2_solutions.ipynb` relies on exactly that file's
season-mismatch/cloud-gap flaws as a teaching point (a naive classifier overestimating real land-use
change). `negros_imagery.nc` is for Part II's plain eyeball comparison only.

In [ ]:
NEGROS_BBOX = [122.3, 9.8, 123.4, 11.0]   # W, S, E, N
TARGET_RES_M = 100

# --- Is Jun-Aug (JJA) or Feb-Apr (FMA) actually less cloudy over Negros? confirmed once, hardcoded here ---
# FMA wins clearly (median cloud% ~18 vs ~53): this is the Philippine dry season ("tag-init"), not JJA (habagat)

s2_items, l5_items = search_seasonal_scenes(
    NEGROS_BBOX, s2_years=range(2022, 2025), l5_years=range(1990, 1997),
    season=("02-01", "04-30"), s2_cloud_lt=60, l5_cloud_lt=70,
)
print(f"Sentinel-2 L2A: {len(s2_items)} scenes pooled (Feb-Apr 2022-2024, cloud<60%)")
print(f"Landsat 5 TM:   {len(l5_items)} scenes pooled (Feb-Apr 1990-1996, cloud<70%)")

In [ ]:
# Per-pixel cloud/shadow-masked temporal median composite for both sensors, on one shared grid
composite = build_landsat_sentinel_composite(NEGROS_BBOX, s2_items, l5_items, target_res_m=TARGET_RES_M)
s2_composite, l5_composite = composite["s2"], composite["l5"]
LAT, LON = composite["grid"]["lat"], composite["grid"]["lon"]

for label, comp in [("S2", s2_composite), ("L5", l5_composite)]:
    for b, arr in comp.items():
        print(f"  [{label}] {b}: {np.isnan(arr).mean():.1%} NaN in composite")

In [ ]:
# Reuse the existing S2 composite (156 scenes, Feb-Apr 2022-2024, cloud<60% -- unchanged from a
# prior run) instead of rebuilding it; only Landsat needed reprocessing today.
existing = xr.open_dataset(RAW_DIR / "Sentinel2" / "negros_imagery.nc")
s2_slice = existing.sel(time=existing.time.max())  # the S2 (recent) time slice
s2_composite = {b: s2_slice[b].values for b in ["red", "green", "blue", "nir"]}
LAT, LON = existing.latitude.values, existing.longitude.values
existing.close()

l5_cache = np.load(RAW_DIR / "Sentinel2" / "l5_composite_cache.npz")
l5_composite = {b: l5_cache[f"l5_{b}"] for b in ["red", "green", "blue", "nir"]}

for label, comp in [("S2", s2_composite), ("L5", l5_composite)]:
    for b, arr in comp.items():
        print(f"  [{label}] {b}: {np.isnan(arr).mean():.1%} NaN in composite")

In [ ]:
# Combine both periods into a single NetCDF with a 'time' dimension
def to_uint8_stretch(band, vmax=0.3):
    """simple linear reflectance stretch (0-vmax -> 0-255) for true-colour display"""
    return np.clip(band, 0, vmax) / vmax * 255


def build_period_vars(comp):
    out = {b: (("latitude", "longitude"), comp[b]) for b in ["red", "green", "blue", "nir"]}
    out["visual_r"] = (("latitude", "longitude"), to_uint8_stretch(comp["red"]))
    out["visual_g"] = (("latitude", "longitude"), to_uint8_stretch(comp["green"]))
    out["visual_b"] = (("latitude", "longitude"), to_uint8_stretch(comp["blue"]))
    return out


def median_year(items):
    return int(np.median([pd.Timestamp(it.properties["datetime"][:10]).year for it in items]))


l5_time = pd.Timestamp(f"{median_year(l5_items)}-03-15")   # nominal mid-dry-season representative date
s2_time = pd.Timestamp(f"{median_year(s2_items)}-03-15")

ds_hist = xr.Dataset(build_period_vars(l5_composite), coords={"latitude": LAT, "longitude": LON}).expand_dims(time=[l5_time])
ds_recent = xr.Dataset(build_period_vars(s2_composite), coords={"latitude": LAT, "longitude": LON}).expand_dims(time=[s2_time])

combined = xr.concat([ds_hist, ds_recent], dim="time")
for v in combined.data_vars:
    combined[v] = combined[v].astype("float32")

combined.attrs["description"] = (
    "Cloud-free, season-matched composite imagery over Negros: Landsat 5 TM (historical) and "
    "Sentinel-2 L2A (recent), each median-composited from many Feb-Apr (Philippine dry season) "
    "scenes pooled across several years, to suppress cloud/shadow gaps and remove wet/dry-season "
    "NDVI bias from the historical-vs-recent comparison."
)
combined.attrs["historical_source"] = f"Landsat 5 TM, {len(l5_items)} scenes, Feb-Apr 1990-1996, cloud<70%"
combined.attrs["recent_source"] = f"Sentinel-2 L2A, {len(s2_items)} scenes, Feb-Apr 2022-2024, cloud<60%"
combined.attrs["bands_note"] = (
    "red/green/blue/nir are physical surface reflectance (0-1 float; NDVI=(nir-red)/(nir+red) works "
    "directly on these); NaN means no clear observation existed anywhere in the pooled scenes for that "
    "pixel; visual_r/g/b are a simple 0-0.3 reflectance stretch to 0-255 for true-colour display"
)
combined.attrs["time_note"] = (
    "time coordinates are nominal representative dates (median year of the pooled scenes, mid-dry-season) "
    "-- each time slice is a multi-year median composite, not a single acquisition"
)

out_nc = RAW_DIR / "Sentinel2" / "negros_imagery.nc"
out_nc.parent.mkdir(parents=True, exist_ok=True)
combined.to_netcdf(out_nc)
print(f"Saved: {out_nc}")
print(combined)

In [ ]:
# --- Exercise-ready version: coarsen ~2x (100m -> ~200m) to keep the file small ---
day2_dir = PROCESSED_DIR / "day2"
day2_dir.mkdir(parents=True, exist_ok=True)

ds_coarse = combined.coarsen(latitude=2, longitude=2, boundary="trim").mean()
for v in ds_coarse.data_vars:
    ds_coarse[v] = ds_coarse[v].astype("float32")
ds_coarse.attrs = combined.attrs
ds_coarse.attrs["resolution_note"] = "coarsened ~2x from the ~100m native composite for a lighter workshop file (~200m)"

out_path = day2_dir / "negros_imagery.nc"
ds_coarse.to_netcdf(out_path)
print(f"Saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

### 3b. High-resolution Guimaras island composite (exercises)

Section 3 above builds a workshop-wide comparison at ~200 m over all of Negros. Some exercises
want a much sharper look at a specific spot on neighbouring Guimaras island instead:

- **Visual (true-colour) exercise**: needs full Sentinel-2 native resolution (10 m) so land-cover
  boundaries are actually visible when zoomed in, but only over a small area -- a forest-loss
  hotspot identified from the Hansen data in Part IV of `day2_solutions.ipynb`.
- **Classification exercise**: needs the same historical/recent comparison but over the whole
  island, at a resolution students can classify without needing gigabytes of RAM -- and doesn't
  need RGB display bands at all, only nir/red for NDVI.

`NEGROS_BBOX` already fully contains Guimaras, and both sensors' footprints are wide enough that a
fresh, tighter search over just Guimaras returns far fewer scenes than the Negros-wide pool (31 S2,
12 L5, vs. 156/63). Even so, holding `(n_items, height, width)` float32 stacks for all 4 bands at
10 m native resolution over an island-sized bbox (~13M pixels/band) would run into tens of GB of
RAM in `_composite_period()`, which keeps every pooled scene's masked band in memory until the
final per-pixel median. So we keep only the `N_SCENES` least-cloudy scenes per sensor -- Guimaras
is small enough that a couple dozen scenes still fully cloud/gap-fill it. Runtime is dominated by
per-scene reprojection cost onto the ~13M-pixel output grid (no decimation happens at 10 m target
res on already-10m-native S2), not network transfer -- expect ~45s/scene for S2, a few sec/scene
for L5, so several minutes total even with the reduced scene count.

Output:
- `data/raw/Sentinel2/guimaras_imagery.nc` -- full composite, 10 m, all bands (~725 MB, raw master)
- `data/processed/day2/guimaras_zoom_imagery.nc` -- cropped to the forest-loss hotspot bbox, 10 m,
  all bands (~2 MB) -- true-colour visual exercise
- `data/processed/day2/guimaras_imagery.nc` -- full island, coarsened to ~30 m, nir/red only
  (~23 MB) -- NDVI / land-cover classification exercise

In [ ]:
GUIMARAS_BBOX = [122.46, 10.4, 122.75, 10.76]   # W, S, E, N -- checked against the land-sea mask
GUIMARAS_RES_M = 10                             # full Sentinel-2 native sharpness
N_SCENES = 25                                   # least-cloudy scenes per sensor -- see markdown above


def least_cloudy(items, n):
    return sorted(items, key=lambda it: it.properties.get("eo:cloud_cover", 100))[:n]


s2_items_g_all, l5_items_g_all = search_seasonal_scenes(
    GUIMARAS_BBOX, s2_years=range(2022, 2025), l5_years=range(1990, 1997),
    season=("02-01", "04-30"), s2_cloud_lt=60, l5_cloud_lt=70,
)
print(f"Sentinel-2: {len(s2_items_g_all)} pooled over Guimaras bbox")
print(f"Landsat 5:  {len(l5_items_g_all)} pooled over Guimaras bbox")

s2_items_g = least_cloudy(s2_items_g_all, N_SCENES)
l5_items_g = least_cloudy(l5_items_g_all, N_SCENES)
print(f"Using {len(s2_items_g)} least-cloudy S2 scenes, {len(l5_items_g)} least-cloudy L5 scenes")

In [ ]:
# Per-pixel cloud/shadow-masked temporal median composite, same recipe as Section 3 but over
# Guimaras at full 10m S2 resolution -- expect several minutes (see markdown above)
composite_g = build_landsat_sentinel_composite(GUIMARAS_BBOX, s2_items_g, l5_items_g, target_res_m=GUIMARAS_RES_M)
s2_composite_g, l5_composite_g = composite_g["s2"], composite_g["l5"]
LAT_G, LON_G = composite_g["grid"]["lat"], composite_g["grid"]["lon"]

for label, comp in [("S2", s2_composite_g), ("L5", l5_composite_g)]:
    for b, arr in comp.items():
        print(f"  [{label}] {b}: {np.isnan(arr).mean():.1%} NaN in composite")

In [ ]:
# Combine into a time-dimensioned NetCDF, same structure as Section 3's `combined`
# (reuses to_uint8_stretch/build_period_vars/median_year defined in the d2-s2-nc cell above)
l5_time_g = pd.Timestamp(f"{median_year(l5_items_g)}-03-15")
s2_time_g = pd.Timestamp(f"{median_year(s2_items_g)}-03-15")

ds_hist_g = xr.Dataset(build_period_vars(l5_composite_g), coords={"latitude": LAT_G, "longitude": LON_G}).expand_dims(time=[l5_time_g])
ds_recent_g = xr.Dataset(build_period_vars(s2_composite_g), coords={"latitude": LAT_G, "longitude": LON_G}).expand_dims(time=[s2_time_g])

combined_g = xr.concat([ds_hist_g, ds_recent_g], dim="time")
for v in combined_g.data_vars:
    combined_g[v] = combined_g[v].astype("float32")

combined_g.attrs["description"] = (
    "High-resolution (10 m) cloud-free, season-matched composite imagery over Guimaras island: "
    "Landsat 5 TM (historical) and Sentinel-2 L2A (recent), for visual-comparison exercises. "
    "Built at full Sentinel-2 native resolution; Landsat 5 (native 30 m) is oversampled onto this "
    "grid, so it carries no genuine detail below ~30 m -- intended for eyeballing, not analysis."
)
combined_g.attrs["historical_source"] = f"Landsat 5 TM, {len(l5_items_g)} least-cloudy scenes (of {len(l5_items_g_all)} pooled), Feb-Apr 1990-1996, cloud<70%"
combined_g.attrs["recent_source"] = f"Sentinel-2 L2A, {len(s2_items_g)} least-cloudy scenes (of {len(s2_items_g_all)} pooled), Feb-Apr 2022-2024, cloud<60%"
combined_g.attrs["bands_note"] = combined.attrs["bands_note"]
combined_g.attrs["time_note"] = combined.attrs["time_note"]

out_nc_g = RAW_DIR / "Sentinel2" / "guimaras_imagery.nc"
combined_g.to_netcdf(out_nc_g)
print(f"Saved {out_nc_g}  ({out_nc_g.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# --- Exercise 1: small 10m true-colour crop over a Hansen-identified forest-loss hotspot ---
GUIMARAS_ZOOM_BBOX = dict(longitude=slice(122.63, 122.65), latitude=slice(10.64, 10.625))
zoom_g = combined_g.sel(**GUIMARAS_ZOOM_BBOX)
zoom_g.attrs["description"] = (
    combined_g.attrs["description"] + " Cropped to a small forest-loss hotspot near Guimaras "
    "(122.63-122.65E, 10.625-10.64N) at full 10 m native resolution for the true-colour visual exercise."
)

zoom_path = day2_dir / "guimaras_zoom_imagery.nc"
zoom_g.to_netcdf(zoom_path)
print(f"Saved {zoom_path}  ({zoom_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# --- Exercise 1b: same recipe, second forest-loss hotspot (cleared for agriculture, confirmed on Google Maps) ---
GUIMARAS_ZOOM_BBOX_CASE2 = dict(longitude=slice(122.585, 122.595), latitude=slice(10.6525, 10.64))
combined_g_raw = xr.open_dataset(RAW_DIR / "Sentinel2" / "guimaras_imagery.nc")
zoom_g_case2 = combined_g_raw.sel(**GUIMARAS_ZOOM_BBOX_CASE2)
zoom_g_case2.attrs["description"] = (
    combined_g_raw.attrs["description"] + " Cropped to a second forest-loss hotspot near Guimaras "
    "(122.585-122.595E, 10.6525-10.64N, cleared for agriculture) at full 10 m native resolution for the true-colour visual exercise."
)

zoom_path_case2 = day2_dir / "guimaras_zoom_imagery_case2.nc"
zoom_g_case2.to_netcdf(zoom_path_case2)
print(f"Saved {zoom_path_case2}  ({zoom_path_case2.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# --- Exercise 2: full-island nir/red only, coarsened 10m -> ~30m, for NDVI / land-cover classification ---
classif_g = combined_g[["nir", "red"]].coarsen(latitude=3, longitude=3, boundary="trim").mean()
for v in classif_g.data_vars:
    classif_g[v] = classif_g[v].astype("float32")

classif_g.attrs["description"] = (
    "Cloud-free, season-matched Landsat 5 / Sentinel-2 composite imagery over Guimaras island, "
    "coarsened 10m -> ~30m and reduced to nir/red bands only (sufficient for NDVI / land-cover "
    "classification exercises; RGB display bands were dropped to keep the file small)."
)
classif_g.attrs["historical_source"] = combined_g.attrs["historical_source"]
classif_g.attrs["recent_source"] = combined_g.attrs["recent_source"]
classif_g.attrs["bands_note"] = (
    "nir/red are physical surface reflectance (0-1 float; NDVI=(nir-red)/(nir+red) works directly "
    "on these); NaN means no clear observation existed anywhere in the pooled scenes for that pixel"
)
classif_g.attrs["time_note"] = combined_g.attrs["time_note"]
classif_g.attrs["resolution_note"] = "coarsened ~3x from the 10m native composite (guimaras_imagery.nc in data/raw) for a lighter workshop file (~30m)"

classif_path = day2_dir / "guimaras_imagery.nc"
classif_g.to_netcdf(classif_path)
print(f"Saved {classif_path}  ({classif_path.stat().st_size / 1e6:.2f} MB)")

---
## 4. Hansen Global Forest Change — Negros

Hansen et al. (2013) provide annual forest cover and loss data from 2000 to present
at 30 m resolution as public COGs on Google Cloud Storage.

Layers:
- `treecover2000` — canopy cover in year 2000 (%)
- `lossyear` — year of first forest loss (1=2001 … 23=2023, 0=no loss)

Negros (9–11°N) straddles two 10°×10° tiles (`20N_120E` and `10N_120E`).
We use rasterio windowed reads on the remote COGs — only the Negros-sized patch
is transferred, not the full global tile.

In [ ]:
HANSEN_LAYERS = ['treecover2000', 'lossyear']
HANSEN_TILES  = ['20N_120E', '10N_120E']   # northern tile first (higher latitude)
NEGROS_FULL   = (121.5, 8.8, 124.5, 11.5)  # W, S, E, N — slightly wider than NEGROS_BBOX

out_dir = RAW_DIR / 'Hansen'
hansen_paths = download_hansen_tiles(NEGROS_FULL, HANSEN_TILES, HANSEN_LAYERS, out_dir=out_dir)
print(hansen_paths)

In [ ]:
# --- Build exercise-ready netCDF: crop tight to NEGROS_BBOX, reproject to plain lat/lon, compact dtypes ---
day2_dir = PROCESSED_DIR / "day2"
day2_dir.mkdir(parents=True, exist_ok=True)

layer_arrays = {}
for layer in HANSEN_LAYERS:
    da = rioxarray.open_rasterio(hansen_paths[layer]).squeeze("band", drop=True)
    layer_arrays[layer] = da

hansen_ds = xr.Dataset(layer_arrays).rename({"x": "longitude", "y": "latitude"})
hansen_ds = hansen_ds.sel(longitude=slice(NEGROS_BBOX[0], NEGROS_BBOX[2]),
                           latitude=slice(NEGROS_BBOX[3], NEGROS_BBOX[1]))
hansen_ds["treecover2000"] = hansen_ds["treecover2000"].astype("uint8")
hansen_ds["lossyear"] = hansen_ds["lossyear"].astype("uint8")
hansen_ds.attrs["description"] = "Hansen Global Forest Change v1.11 (2023), cropped to Negros, native ~30m resolution"
hansen_ds.attrs["source"] = "Hansen et al. (2013), https://storage.googleapis.com/earthenginepartners-hansen/"
hansen_ds.attrs["lossyear_note"] = "year of first forest loss: 1=2001 ... 23=2023, 0=no loss detected"

out_nc = day2_dir / "negros_forest_change.nc"
hansen_ds.to_netcdf(out_nc)
print(f"Saved {out_nc}  ({out_nc.stat().st_size / 1e6:.1f} MB)")
print(hansen_ds)